In [6]:
import pandas as pd

df_test = pd.read_csv("test_transformed.csv")
df_train = pd.read_csv("train_transformed.csv")

df_test.head()

,A1_Score,A2_Score,A3_Score,A4_Score,A5_Score,A6_Score,A7_Score,A8_Score,A9_Score,A10_Score,age,gender,ethnicity,jaundice,austim,contry_of_res,used_app_before,result
0,1,1,0,0,1,1,0,0,1,1,-0.723383,1,10,1,0,14,0,0.793428
1,1,0,0,0,0,0,0,1,0,0,0.076414,1,0,0,0,21,0,-0.451187
2,1,1,1,0,1,1,0,1,1,1,0.384605,1,10,1,0,10,0,-1.168682
3,0,0,0,0,0,0,0,0,0,0,-0.048710,1,9,0,0,14,0,-1.372993
4,0,0,0,1,0,0,0,0,0,0,-1.173700,1,9,0,0,17,0,-0.302103


In [9]:
df_train.shape

(800, 19)

In [10]:
df_test.shape

(200, 18)

In [11]:
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
# Before (with StandardScaler)
scaler = StandardScaler()

# Pipeline including StandardScaler
model_pipeline = Pipeline([
    ('scaler', scaler),
    # other steps like classifier
])
model_pipeline.fit_transform(df_test)


array([[ 0.85972695,  0.89543386, -0.95118973, ..., -0.60698839,
        -0.20412415,  0.79342772],
       [ 0.85972695, -1.11677706, -0.95118973, ...,  0.09184693,
        -0.20412415, -0.45118742],
       [ 0.85972695,  0.89543386,  1.05131497, ..., -1.00632285,
        -0.20412415, -1.16868163],
       ...,
       [ 0.85972695, -1.11677706, -0.95118973, ...,  0.29151416,
        -0.20412415, -1.10093563],
       [-1.16316   ,  0.89543386, -0.95118973, ...,  1.1900167 ,
        -0.20412415, -0.65489244],
       [ 0.85972695, -1.11677706, -0.95118973, ..., -1.10615647,
        -0.20412415,  0.18339173]])

In [ ]:
import pandas as pd
import logging
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Configure logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# Load the dataset
def load_dataset(file_path):
    """Loads the dataset and returns dataframe"""
    try:
        df = pd.read_csv(file_path)
        logging.info(f"Dataset loaded: {file_path}, Shape: {df.shape}")
        return df
    except Exception as e:
        logging.error(f"Error loading dataset from {file_path}: {e}")
        raise

# Define preprocessing pipeline without Imputer
def create_preprocessing_pipeline(numerical_features, categorical_features):
    """Creates a preprocessing pipeline for numerical and categorical features without imputation"""
    logging.info("Creating preprocessing pipeline...")

    # Numerical Feature preprocessing (without imputation)
    numerical_transformer = Pipeline(steps=[
        ('scaler', StandardScaler())  # Scaling numerical features
    ])

    # Categorical Feature preprocessing (without imputation)
    categorical_transformer = Pipeline(steps=[
        ('onehot', OneHotEncoder(handle_unknown='ignore'))  # One-hot encode categorical data
    ])

    # Combine preprocessing steps
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', numerical_transformer, numerical_features),
            ('cat', categorical_transformer, categorical_features)
        ])

    return preprocessor

# Define the full model pipeline
def create_model_pipeline(preprocessor):
    """Creates a complete pipeline with preprocessing and a classifier"""
    logging.info("Creating model pipeline...")
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', RandomForestClassifier(random_state=42))  # Classifier model
    ])
    return pipeline

# Train the model
def train_model(pipeline, X_train, y_train):
    """Trains the model pipeline"""
    logging.info("Training the model...")
    pipeline.fit(X_train, y_train)
    logging.info("Model training completed.")

# Evaluate the model
def evaluate_model(pipeline, X_test, y_test):
    """Evaluates the model on the test data"""
    logging.info("Evaluating the model...")

    # Predict on the test set
    y_pred = pipeline.predict(X_test)

    # Print classification report
    logging.info("Classification Report:\n")
    print(classification_report(y_test, y_pred))

    # Model performance evaluation metrics
    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, average='binary', zero_division=1)
    recall = recall_score(y_test, y_pred, average='binary', zero_division=1)
    f1 = f1_score(y_test, y_pred, average='binary', zero_division=1)
    conf_matrix = confusion_matrix(y_test, y_pred)

    # Log the metrics
    logging.info(f"Accuracy: {accuracy:.4f}")
    logging.info(f"Precision: {precision:.4f}")
    logging.info(f"Recall: {recall:.4f}")
    logging.info(f"F1-Score: {f1:.4f}")
    logging.info(f"Confusion Matrix:\n{conf_matrix}")

    # Optionally, print out the confusion matrix for better readability
    print("\nConfusion Matrix:")
    print(conf_matrix)

# Main Function to train the model
def train_model_pipeline(train_file):
    """Executes the training part of the pipeline"""
    logging.info("Starting the training pipeline execution...")

    # Load the train data
    train_df = load_dataset(train_file)

    # Clean column names (remove leading/trailing spaces)
    train_df.columns = train_df.columns.str.strip()

    # Define target column
    target_column = 'Class/ASD'

    # Check if the target column exists in the dataset
    if target_column not in train_df.columns:
        logging.error(f"Target column '{target_column}' not found in the train dataset!")
        raise KeyError(f"Target column '{target_column}' not found in the train dataset!")

    # Define features and target
    X_train = train_df.drop(columns=[target_column])
    y_train = train_df[target_column]

    # Log the shapes of train data
    logging.info(f"Training data shape: {X_train.shape}")

    # Define feature types
    numerical_features = ['age', 'result']
    categorical_features = ['gender', 'ethnicity', 'jaundice', 'austim', 'contry_of_res', 'used_app_before']

    # Create preprocessing pipeline
    preprocessor = create_preprocessing_pipeline(numerical_features, categorical_features)

    # Create model pipeline
    model_pipeline = create_model_pipeline(preprocessor)

    # Train the model
    train_model(model_pipeline, X_train, y_train)

    return model_pipeline

# Main Function to evaluate the model (Test Data)
def evaluate_model_pipeline(test_file, model_pipeline):
    """Executes the evaluation part of the pipeline"""
    logging.info("Starting the evaluation pipeline execution...")

    # Load the test data
    test_df = load_dataset(test_file)

    # Clean column names (remove leading/trailing spaces)
    test_df.columns = test_df.columns.str.strip()

    # Define features and target
    X_test = test_df.drop(columns=['Class/ASD'], errors='ignore')

    # Check if target column exists for evaluation
    if 'Class/ASD' in test_df.columns:
        y_test = test_df['Class/ASD']
    else:
        y_test = None
        logging.warning("No target column found in test data. Only predictions will be made.")

    # Log the shapes of test data
    logging.info(f"Test data shape: {X_test.shape}")

    # Evaluate the model
    if y_test is not None:
        evaluate_model(model_pipeline, X_test, y_test)
    else:
        predictions = model_pipeline.predict(X_test)
        logging.info("Predictions made on test data:")
        print(predictions)

# Example usage
if __name__ == "__main__":
    train_file = 'train_transformed.csv'
    test_file = 'test_transformed.csv'

    # Train the model
    model_pipeline = train_model_pipeline(train_file)

    # Evaluate the model
    evaluate_model_pipeline(test_file, model_pipeline)


In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import joblib

# Function to preprocess the dataset
def preprocess_data(file_path, target_column):
    """
    Preprocess the dataset by reading it, handling missing values, and splitting features and target.
    :param file_path: Path to the dataset file (CSV format).
    :param target_column: Name of the target column.
    :return: features (X) and target (y) as separate DataFrames.
    """
    data = pd.read_csv(file_path)
    data.dropna(inplace=True)

    if target_column not in data.columns:
        raise KeyError(f"Target column '{target_column}' not found in the dataset!")

    X = data.drop(columns=[target_column])
    y = data[target_column]

    return X, y

# Function to train the model
def train_model_pipeline(train_file, target_column):
    """
    Train a Random Forest model using the training dataset and save the pipeline to a file.
    :param train_file: Path to the training dataset file (CSV format).
    :param target_column: Name of the target column.
    """
    # Preprocess the training data
    X, y = preprocess_data(train_file, target_column)

    # Split the data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

    # Train the model
    model = RandomForestClassifier(random_state=42)
    model.fit(X_train, y_train)

    # Validate the model
    y_val_pred = model.predict(X_val)
    accuracy = accuracy_score(y_val, y_val_pred)
    print("Validation Accuracy:", accuracy)
    print("Validation Classification Report:\n", classification_report(y_val, y_val_pred))

    # Save the trained model to a file
    joblib.dump(model, 'model_pipeline.pkl')
    print("Model pipeline saved to 'model_pipeline.pkl'.")

# Function to evaluate the model
def evaluate_model_pipeline(test_file):
    """
    Load the saved model pipeline and evaluate it using the test dataset.
    :param test_file: Path to the test dataset file (CSV format).
    """
    # Load the test data
    test_data = pd.read_csv(test_file)
    test_data.dropna(inplace=True)

    # Load the saved model pipeline
    model = joblib.load('model_pipeline.pkl')

    # Predict on test data
    predictions = model.predict(test_data)
    print("Predictions on test data:", predictions)

    # Save predictions to a file
    test_data['Predictions'] = predictions
    test_data.to_csv('test_predictions.csv', index=False)
    print("Predictions saved to 'test_predictions.csv'.")

# Example usage
if __name__ == "__main__":
    train_file = "train_transformed.csv"  # Replace with your training dataset path
    test_file = "test_transformed.csv"    # Replace with your test dataset path
    target_column = "Class/ASD"       # Replace with your target column name

    # Train the model
    train_model_pipeline(train_file, target_column)

    # Evaluate the model
    evaluate_model_pipeline(test_file)


Validation Accuracy: 0.85625
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.89      0.94      0.91       124
           1       0.72      0.58      0.65        36

    accuracy                           0.86       160
   macro avg       0.80      0.76      0.78       160
weighted avg       0.85      0.86      0.85       160

Model pipeline saved to 'model_pipeline.pkl'.
Predictions on test data: [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 1 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 1
 0 1 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 1 0 0 0 0 0 0 0 0 0 0 1 0
 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0]
Predictions saved to 'test_predictions.csv'.


In [14]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import GridSearchCV
import joblib

# Function to preprocess the dataset
def preprocess_data(file_path, target_column):
    """
    Preprocess the dataset by reading it, handling missing values, and splitting features and target.
    :param file_path: Path to the dataset file (CSV format).
    :param target_column: Name of the target column.
    :return: features (X) and target (y) as separate DataFrames.
    """
    data = pd.read_csv(file_path)
    data.dropna(inplace=True)

    if target_column not in data.columns:
        raise KeyError(f"Target column '{target_column}' not found in the dataset!")

    X = data.drop(columns=[target_column])
    y = data[target_column]

    return X, y

# Function to train the model
def train_model_pipeline(train_file, target_column):
    """
    Train a Random Forest model using the training dataset and save the pipeline to a file.
    :param train_file: Path to the training dataset file (CSV format).
    :param target_column: Name of the target column.
    """
    # Preprocess the training data
    X, y = preprocess_data(train_file, target_column)

    # Handle class imbalance using SMOTE
    smote = SMOTE(random_state=42)
    X_resampled, y_resampled = smote.fit_resample(X, y)

    # Split the data into training and validation sets
    X_train, X_val, y_train, y_val = train_test_split(X_resampled, y_resampled, test_size=0.2, random_state=42)

    # Define the model and hyperparameter grid
    model = RandomForestClassifier(random_state=42)
    param_grid = {
        'n_estimators': [100, 200],
        'max_depth': [None, 10, 20],
        'min_samples_split': [2, 5]
    }

    # Perform Grid Search
    grid_search = GridSearchCV(model, param_grid, cv=3, scoring='accuracy')
    grid_search.fit(X_train, y_train)

    # Get the best model
    best_model = grid_search.best_estimator_

    # Validate the model
    y_val_pred = best_model.predict(X_val)
    accuracy = accuracy_score(y_val, y_val_pred)
    print("Validation Accuracy:", accuracy)
    print("Validation Classification Report:\n", classification_report(y_val, y_val_pred))

    # Save the trained model to a file
    joblib.dump(best_model, 'model_pipeline.pkl')
    print("Model pipeline saved to 'model_pipeline.pkl'.")

# Function to evaluate the model
def evaluate_model_pipeline(test_file):
    """
    Load the saved model pipeline and evaluate it using the test dataset.
    :param test_file: Path to the test dataset file (CSV format).
    """
    # Load the test data
    test_data = pd.read_csv(test_file)
    test_data.dropna(inplace=True)

    # Load the saved model pipeline
    model = joblib.load('model_pipeline.pkl')

    # Predict on test data
    predictions = model.predict(test_data)
    print("Predictions on test data:", predictions)

    # Save predictions to a file
    test_data['Predictions'] = predictions
    test_data.to_csv('test_predictions.csv', index=False)
    print("Predictions saved to 'test_predictions.csv'.")

# Example usage
if __name__ == "__main__":
    train_file = "train_transformed.csv"  # Replace with your training dataset path
    test_file = "test_transformed.csv"    # Replace with your test dataset path
    target_column = "Class/ASD"       # Replace with your target column name

    # Train the model
    train_model_pipeline(train_file, target_column)

    # Evaluate the model
    evaluate_model_pipeline(test_file)


Validation Accuracy: 0.875
Validation Classification Report:
               precision    recall  f1-score   support

           0       0.92      0.82      0.86       125
           1       0.84      0.93      0.88       131

    accuracy                           0.88       256
   macro avg       0.88      0.87      0.87       256
weighted avg       0.88      0.88      0.87       256

Model pipeline saved to 'model_pipeline.pkl'.
Predictions on test data: [0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 1 1 1 0 0 1 1 1 1 0 0 0 1 0 1 0 0 0 0 1 1 1
 1 0 0 0 1 0 0 0 0 0 1 0 0 1 0 1 1 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0
 0 0 0 1 1 0 1 0 1 0 0 0 0 0 0 0 1 0 0 0 0 0 1 0 1 0 0 0 0 1 0 0 0 0 0 0 1
 0 1 0 1 0 0 0 0 1 0 0 0 0 0 1 0 0 0 0 0 0 0 0 1 0 0 0 1 0 0 0 0 0 0 0 1 0
 1 0 0 0 0 0 0 0 0 1 1 0 0 1 0 0 1 0 1 1 0 0 1 1 0 0 0 0 0 1 0 0 0 0 0 0 0
 1 0 0 1 0 0 1 0 0 0 0 0 0 0 0]
Predictions saved to 'test_predictions.csv'.
